In [1]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 388.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.3 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 4.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 MB 1.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 891.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 5.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 6.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: typing-extension

In [2]:
import os
import time
import cv2
import numpy as np

try:
    import ipywidgets as widgets
    from IPython.display import display
    USE_WIDGETS = True
except ImportError:
    USE_WIDGETS = False

from ultralytics import YOLO
from Arm_Lib import Arm_Device

# ---------------- DOFBOT SETUP ----------------
Arm = Arm_Device()
time.sleep(0.1)
print("✅ DOFBOT connected")

# ---------------- YOLO MODEL ------------------
DETECTION_WEIGHTS = "best.pt"   # CHANGE PATH IF NEEDED
det_model = YOLO(DETECTION_WEIGHTS)

print("✅ YOLO detection model loaded:", DETECTION_WEIGHTS)
print("Model classes:", det_model.names)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ DOFBOT connected
✅ YOLO detection model loaded: best.pt
Model classes: {0: 'bottle', 1: 'can', 2: 'detergent', 3: 'pulses', 4: 'seafood', 5: 'fruit'}


In [3]:
# =========================================================
#   ARM CONTROL HELPERS (from clamp_block.ipynb)
# =========================================================

def arm_clamp_block(enable: int):
    """Open/close gripper."""
    if enable == 0:
        Arm.Arm_serial_servo_write(6, 60, 400)
    else:
        Arm.Arm_serial_servo_write(6, 135, 400)
    time.sleep(0.5)

def arm_move(p, s_time=500):
    """Move DOFBOT servos 1–5 to angles in list p."""
    for i in range(5):
        _id = i + 1
        if _id == 5:
            time.sleep(.1)
            Arm.Arm_serial_servo_write(_id, p[i], int(s_time * 1.2))
        else:
            Arm.Arm_serial_servo_write(_id, p[i], s_time)
        time.sleep(.01)
    time.sleep(s_time / 1000.0)

def arm_move_up():
    """Raise to safe height."""
    Arm.Arm_serial_servo_write(2, 90, 1500)
    Arm.Arm_serial_servo_write(3, 90, 1500)
    Arm.Arm_serial_servo_write(4, 90, 1500)
    time.sleep(.1)

# =========================================================
#              COORDINATES (YOUR ORIGINALS)
# =========================================================

p_mould  = [90, 50, 50, 10, 90]      # RESTING POSITION (from clamp_block)
p_top    = [90, 80, 50, 50, 270]
p_Brown  = [90, 30, 61, 30, 270]    # PICK AREA

# Drop zones
p_Yellow = [65, 22, 64, 56, 270]    # bottle
p_Red    = [124, 22, 66, 45, 270]   # can
p_Purple = [136, 66, 20, 29, 270]   # detergent
p_Blue   = [44, 66, 20, 28, 270]    # fruit
p_Pink   = [8, 72, 27, 6, 270]      # pulses
p_Navy   = [178, 79, 19, 0, 270]    # seafood

# =========================================================
#           RESTING POSITION — FINAL VERSION
# =========================================================

def go_idle():
    """Return DOFBOT to EXACT clamp_block resting pose."""
    arm_clamp_block(0)                 # Open gripper
    arm_move_up()                      # Lift first (safe)
    arm_move(p_mould, 1000)            # Go to resting pose
    print("🤖 Arm returned to clamp_block resting position.")

# =========================================================
#                CLASS → TARGET BIN MAPPING
# =========================================================

CLASS_TO_TARGET_POSE = {
    "bottle":    p_Yellow,
    "can":       p_Red,
    "detergent": p_Purple,
    "fruit":     p_Blue,
    "pulses":    p_Pink,
    "seafood":   p_Navy,
}

def pick_and_place(target_pose):
    """Pick from grey zone → place into bin → return to idle."""
    # go above pick
    arm_move(p_top, 1000)
    arm_move(p_Brown, 1000)
    arm_clamp_block(1)  # grab

    # move to bin
    arm_move(p_top, 1000)
    arm_move(target_pose, 1000)
    arm_clamp_block(0)  # release

    # return home
    arm_move_up()
    arm_move(p_mould, 1100)
    print("✅ Pick-and-place complete. Back to resting pose.")


In [4]:
if USE_WIDGETS:
    image_widget = widgets.Image(format='jpeg', width=640, height=480)
    label_widget = widgets.Label(value="Waiting for camera...")
    display(image_widget)
    display(label_widget)
else:
    image_widget = None
    label_widget = None

def debug_list_video_devices(max_index=10):
    print("Checking /dev/video* ...")
    for i in range(max_index):
        dev = f"/dev/video{i}"
        if os.path.exists(dev):
            print(" ", dev)

def find_camera(preferred_index=None, max_index=10):
    candidates = []
    if preferred_index is not None:
        candidates.append(preferred_index)
    candidates.extend(range(max_index))

    for idx in candidates:
        cap = cv2.VideoCapture(idx)
        if cap.isOpened():
            print(f"✅ Camera opened at index {idx}")
            return cap
        cap.release()

    raise RuntimeError("❌ No camera opened!")

def quick_camera_test(preferred_index=0):
    cap = find_camera(preferred_index)
    ok, frame = cap.read()
    if ok:
        if USE_WIDGETS:
            ok, buf = cv2.imencode(".jpg", frame)
            image_widget.value = buf.tobytes()
        print("✅ Camera test OK")
    cap.release()


Image(value=b'', format='jpeg', height='480', width='640')

Label(value='Waiting for camera...')

In [5]:
def get_top_detection(result, model):
    """
    Return (class_name, confidence) of highest-confidence detection.
    If none → (None, 0)
    """
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    top_conf = 0.0
    top_name = None

    for box in result.boxes:
        conf = float(box.conf.item())
        cls_id = int(box.cls.item())
        name = model.names[cls_id]

        if conf > top_conf:
            top_conf = conf
            top_name = name

    return top_name, top_conf


In [6]:
def run_grocery_sorting_loop(
    model,
    conf_threshold=0.5,
    frames_to_confirm=10,
    preferred_cam_index=0
):
    """
    - Robot starts in clamp_block resting pose
    - Runs YOLO detection
    - Confirms same class for N frames
    - Executes pick-and-place
    - Returns to resting pose after each cycle
    """

    go_idle()   # ALWAYS start in resting pose

    cap = find_camera(preferred_cam_index)
    candidate = None
    streak = 0

    print("🚀 Sorting loop started. Interrupt kernel to stop.\n")

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                continue

            results = model(frame, conf=conf_threshold, verbose=False)
            result = results[0]

            top_name, top_conf = get_top_detection(result, model)

            if USE_WIDGETS:
                annotated = result.plot()
                ok, buf = cv2.imencode(".jpg", annotated)
                if ok:
                    image_widget.value = buf.tobytes()
                if label_widget:
                    label_widget.value = f"Detected: {top_name} ({top_conf:.2f})"

            # --- Confirmation logic ---
            if top_name is None:
                candidate = None
                streak = 0
            else:
                if top_name == candidate:
                    streak += 1
                else:
                    candidate = top_name
                    streak = 1

                # --- If confirmed over N frames ---
                if streak >= frames_to_confirm:
                    print(f"\n🎯 Confirmed class: {candidate}")

                    target = CLASS_TO_TARGET_POSE.get(candidate)
                    if target:
                        print(f"➡️ Moving '{candidate}' to bin...")
                        pick_and_place(target)
                    else:
                        print(f"⚠️ No bin mapped for class '{candidate}'")

                    # Reset
                    streak = 0
                    candidate = None

                    print("⏳ Waiting 1.5s...")
                    time.sleep(1.5)
                    go_idle()   # ALWAYS return to clamp_block resting pose

            time.sleep(0.02)

    except KeyboardInterrupt:
        print("\n🛑 Loop interrupted by user.")

    finally:
        cap.release()
        go_idle()
        print("✔️ Camera released. Robot reset.")


In [ ]:
# Adjust preferred_cam_index if needed (0 or 1 usually)
run_grocery_sorting_loop(
    model=det_model,
    conf_threshold=0.5,
    frames_to_confirm=10,
    preferred_cam_index=0
)



🤖 Arm returned to clamp_block resting position.
✅ Camera opened at index 0
🚀 Sorting loop started. Interrupt kernel to stop.


🎯 Confirmed class: detergent
➡️ Moving 'detergent' to bin...
✅ Pick-and-place complete. Back to resting pose.
⏳ Waiting 1.5s...
🤖 Arm returned to clamp_block resting position.

🎯 Confirmed class: seafood
➡️ Moving 'seafood' to bin...
✅ Pick-and-place complete. Back to resting pose.
⏳ Waiting 1.5s...
🤖 Arm returned to clamp_block resting position.
